In [2]:
%pip install tensorflow

In [3]:
import os
import cv2
from PIL import Image
import numpy as np
import tensorflow as tf
import keras
from keras import layers

In [4]:
images = []
labels = []

path = 'D://Thumbs'
target_size = (100, 100)

for folder in ['Bad_balanced', 'Good_valid']:
    filepath = os.path.join(path, folder)
    label = 0 if folder == 'Bad_balanced' else 1  # 0 for Bad, 1 for Good

    for file in os.listdir(filepath):
        if not file.endswith('.png'):
            continue

        img_path = os.path.join(filepath, file)

        try:
            img = Image.open(img_path).convert('L')
            img = img.resize(target_size)
            img = np.array(img)
            img = img.reshape((100, 100, 1))  # shape for CNN
            images.append(img)
            labels.append(label)
        except Exception as e:
            print(f"Skipping {file}: {e}")
            continue

# Convert to numpy arrays
x = np.array(images, dtype='float32') / 255.0  # normalize here or use Rescaling layer
y = np.array(labels)


In [5]:
from sklearn.model_selection import train_test_split

x_train, x_test,y_train, y_test = train_test_split(x, y, test_size=0.2, stratify=y, random_state=42)

In [6]:
# model without augmentation

model  = tf.keras.Sequential([
    layers.Input(shape=(100,100,1)),
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Flatten(),
    layers.Dense(512, activation='relu'),
    layers.Dense(1, activation='sigmoid')
    
])

In [7]:
model.compile(loss = 'binary_crossentropy', optimizer=keras.optimizers.RMSprop(learning_rate=0.01), metrics=['accuracy'])

In [11]:
history = model.fit(x_train, y_train, epochs= 15, validation_data=(x_test, y_test), batch_size=32)

Epoch 1/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 16s 292ms/step - accuracy: 0.5083 - loss: 22.9671 - val_accuracy: 0.7184 - val_loss: 0.4722
Epoch 2/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 13s 278ms/step - accuracy: 0.8140 - loss: 0.4321 - val_accuracy: 0.5000 - val_loss: 2.5357
Epoch 3/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 14s 285ms/step - accuracy: 0.8192 - loss: 0.8423 - val_accuracy: 0.9000 - val_loss: 0.2061
Epoch 4/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 15s 307ms/step - accuracy: 0.9104 - loss: 0.2436 - val_accuracy: 0.9316 - val_loss: 0.1733
Epoch 5/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 14s 299ms/step - accuracy: 0.8977 - loss: 0.2703 - val_accuracy: 0.7053 - val_loss: 0.6360
Epoch 6/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 14s 282ms/step - accuracy: 0.8924 - loss: 0.2933 - val_accuracy: 0.9263 - val_loss: 0.1651
Epoch 7/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 15s 320ms/step - accuracy: 0.9327 - loss: 0.1705 - val_accuracy: 0.9526 - val_loss: 0.1135
Epoch 8/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 16s 326ms/step - accuracy: 0.9434 - loss: 0.1491 - val_acc

In [8]:
#model with augmentation

model_with_augmentation  = tf.keras.Sequential([
    layers.Input(shape=(100,100,1)),
    # layers.RandomFlip('horizontal'), shouldnt be used as it will flipp the finger prints
    layers.RandomRotation(0.02, fill_mode= 'nearest'),
    layers.RandomTranslation(0.02, 0.02, fill_mode= 'nearest'),
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Flatten(),
    layers.Dense(512, activation='relu'),
    layers.Dense(1, activation='sigmoid')
    
])

In [9]:
model_with_augmentation.compile(loss = 'binary_crossentropy', optimizer=keras.optimizers.RMSprop(learning_rate=0.01), metrics=['accuracy'])

In [10]:
history = model_with_augmentation.fit(x_train, y_train, epochs= 35, validation_data=(x_test, y_test), batch_size=32)

Epoch 1/35
 6/48 ━━━━━━━━━━━━━━━━━━━━ 18s 430ms/step - accuracy: 0.4674 - loss: 57.4916

KeyboardInterrupt: 

In [15]:
#model with augmentation using datagenerator

model_with_datagenerator  = tf.keras.Sequential([
    layers.Input(shape=(100,100,1)),
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Flatten(),
    layers.Dense(512, activation='relu'),
    layers.Dense(1, activation='sigmoid')
    
])

In [16]:
model_with_datagenerator.compile(loss = 'binary_crossentropy', optimizer=keras.optimizers.RMSprop(learning_rate=0.01), metrics=['accuracy'])

In [20]:
from keras.callbacks import Callback

class myCallBack(Callback):
    def on_epoch_end(self, epoch, logs=None):
        if logs['accuracy']>0.97:
            print("desired accuracy recieved so cancelling training")
            self.model.stop_training = True


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Define augmentation (NO flipping)
datagen = ImageDataGenerator(
    rotation_range=2,
    width_shift_range=0.02,
    height_shift_range=0.02,
    fill_mode='nearest'
)

# Fit the generator on training data
datagen.fit(x_train)

# Use it during training
model_with_datagenerator.fit(datagen.flow(x_train, y_train, batch_size=32),
          validation_data=(x_test, y_test),
          epochs=30, callbacks=[myCallBack()])


Epoch 1/30
48/48 ━━━━━━━━━━━━━━━━━━━━ 14s 294ms/step - accuracy: 0.7927 - loss: 0.4870 - val_accuracy: 0.8553 - val_loss: 0.3788
Epoch 2/30
48/48 ━━━━━━━━━━━━━━━━━━━━ 13s 277ms/step - accuracy: 0.7626 - loss: 0.6985 - val_accuracy: 0.8526 - val_loss: 0.3780
Epoch 3/30
48/48 ━━━━━━━━━━━━━━━━━━━━ 13s 280ms/step - accuracy: 0.8075 - loss: 0.4093 - val_accuracy: 0.7947 - val_loss: 0.5155
Epoch 4/30
48/48 ━━━━━━━━━━━━━━━━━━━━ 14s 295ms/step - accuracy: 0.8290 - loss: 0.4042 - val_accuracy: 0.8184 - val_loss: 0.5188
Epoch 5/30
48/48 ━━━━━━━━━━━━━━━━━━━━ 14s 298ms/step - accuracy: 0.8692 - loss: 0.3445 - val_accuracy: 0.8658 - val_loss: 0.2859
Epoch 6/30
48/48 ━━━━━━━━━━━━━━━━━━━━ 15s 313ms/step - accuracy: 0.8590 - loss: 0.3225 - val_accuracy: 0.8658 - val_loss: 0.2924
Epoch 7/30
48/48 ━━━━━━━━━━━━━━━━━━━━ 15s 310ms/step - accuracy: 0.8581 - loss: 0.3097 - val_accuracy: 0.9079 - val_loss: 0.2846
Epoch 8/30
48/48 ━━━━━━━━━━━━━━━━━━━━ 15s 306ms/step - accuracy: 0.8756 - loss: 0.2974 - val_accu

In [23]:
model_with_datagenerator.save('ccn_model_datagenerator.keras')